# Gemma 4 Good Hackathon<br>Kaggle Competition

## GPU ACCELERATION

In [ ]:
%load_ext cudf.pandas
# NOTE: cuml.accel is disabled — RAPIDS cu12 JIT compilation fails against
# the system's CUDA 13.1 headers (/opt/cuda/include/cuda_fp8.hpp).
# cuml can still be used via direct imports (e.g., from cuml.ensemble import ...).
# cudf.pandas acceleration works fine for dataframe operations.

# --- Preload CUDA shared libs for TensorFlow + PyTorch GPU support ---
# TF 2.21 needs cu12 libs; PyTorch cu130 needs cu13 libs.
# Both are pip-installed under nvidia/*/lib but the Jupyter process
# won't pick them up via LD_LIBRARY_PATH alone (already running).
# We preload them with ctypes so dlopen() finds them in-process.
import os, site, ctypes

_nv = os.path.join(site.getsitepackages()[0], "nvidia")

# Map of (soname, subdirectory) — cu12 for TensorFlow, cu13 for PyTorch
# NOTE: nvrtc-builtins must be loaded BEFORE libnvrtc, because libnvrtc
# tries to dlopen the builtins lib at load time.
_libs_to_preload = [
    # TensorFlow cu12 libs
    ("cuda_runtime/lib/libcudart.so.12", "TF"),
    ("cublas/lib/libcublas.so.12", "TF"),
    ("cublas/lib/libcublasLt.so.12", "TF"),
    ("cufft/lib/libcufft.so.11", "TF"),
    ("cusolver/lib/libcusolver.so.11", "TF"),
    ("cusparse/lib/libcusparse.so.12", "TF"),
    ("cuda_nvrtc/lib/libnvrtc-builtins.so.12", "TF"),
    ("cuda_nvrtc/lib/libnvrtc.so.12", "TF"),
    ("nvjitlink/lib/libnvJitLink.so.12", "TF"),
    ("cudnn/lib/libcudnn.so.9", "TF+PT"),
    # PyTorch cu13 libs (PyTorch loads most via torch itself,
    # but preloading ensures availability)
    ("cu13/lib/libcudart.so.13", "PT"),
    ("cu13/lib/libcublas.so.13", "PT"),
    ("cu13/lib/libnvrtc-builtins.so.13.0", "PT"),
    ("cu13/lib/libnvrtc.so.13", "PT"),
]

_loaded = []
for _rel, _tag in _libs_to_preload:
    _path = os.path.join(_nv, _rel)
    if os.path.exists(_path):
        try:
            ctypes.CDLL(_path, mode=ctypes.RTLD_GLOBAL)
            _loaded.append(os.path.basename(_rel))
        except OSError:
            pass

# Also add all nvidia lib dirs to LD_LIBRARY_PATH for any remaining lookups
_lib_dirs = [os.path.join(_nv, d, "lib") for d in os.listdir(_nv)
             if os.path.isdir(os.path.join(_nv, d, "lib"))]
os.environ["LD_LIBRARY_PATH"] = ":".join(_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print(f"GPU acceleration configured — preloaded {len(_loaded)} CUDA libs")

## IMPORTS
- **NOTE**: run command `direnv allow` in terminal (within the virtual environment) before running imports cell

In [ ]:
# General/Data Science Imports
import pandas as pd
import numpy as np

# Plotting Imports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots